In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

import nltk
# Download NLTK data once (stopwords, wordnet for lemmatization)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer

# --- Data paths (relative to notebook in model/) ---
DATA_DIR = Path("../10_categories_of_yahoo_answers_for_nlp_tasks_csv")
TRAIN_PATH = DATA_DIR / "train_mini.csv"
TEST_PATH = DATA_DIR / "test.csv"



In [2]:
STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def preprocess_df(df):
    title = df["question_title"].fillna("").astype(str)
    content = df["question_content"].fillna("").astype(str)
    question = (title + " " + content).str.strip()
    return question.apply(normalize_text)

def tokenize_for_vectorizer(text):
    text = text.lower().strip()
    tokens = re.findall(r"\b\w+\b", text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    return tokens


# Load train and test data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# Preprocess text and store in new column 
train_df["text"] = preprocess_df(train_df)
test_df["text"] = preprocess_df(test_df)

# Target: class_index
train_df["class_index"] = pd.to_numeric(train_df["class_index"], errors="coerce").astype("Int64")
test_df["class_index"] = pd.to_numeric(test_df["class_index"], errors="coerce").astype("Int64")

# Clean:Drop rows with missing class 
train_df = train_df.dropna(subset=["class_index"])
test_df = test_df.dropna(subset=["class_index"])

# Create train/validation split from train_df
X_raw = train_df["text"].values
y = train_df["class_index"].astype(int).values

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_raw,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Test set remains the provided test.csv
X_test_raw = test_df["text"].values
y_test = test_df["class_index"].astype(int).values

print("Train size:", len(y_train), "Val size:", len(y_val), "Test size:", len(y_test))
print("Class distribution (train):\n", pd.Series(y_train).value_counts().sort_index())
print("----------")
print("Sample cleaned text (train):")
print(X_train_raw[:3])

Train size: 160000 Val size: 40000 Test size: 59999
Class distribution (train):
 1     16000
2     16000
3     16000
4     16000
5     16000
6     16000
7     16000
8     16000
9     16000
10    16000
Name: count, dtype: int64
----------
Sample cleaned text (train):
<StringArray>
['davinci code what the heck is it all about?? theres a movie out about it, a book, etc but in plain english can you tell me what exactly its about before i waste y money on going to the cinema to watch it?',
                                                                                                                                                                                'hot or not, pig tails (hair)?',
                                                                                                                                                                       'how does electric current cause death?']
Length: 3, dtype: str


In [3]:
# Vectorize text using CountVectorizer on train, then transform val and test
count_vectorizer = CountVectorizer(
    tokenizer=tokenize_for_vectorizer,
    lowercase=False,
    max_features=10000,
)

X_train = count_vectorizer.fit_transform(X_train_raw)
X_val = count_vectorizer.transform(X_val_raw)
X_test = count_vectorizer.transform(X_test_raw)
print("Feature matrix shapes - train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)

# Resulting feature matrix (sparse)
#print(X_train[4])
#print("Sample tokens (first train doc):", tokenize_for_vectorizer(X_train_raw[4])[:25])

/Users/cheukmanzhou/Documents/School/Year4_Sem2/COMPSCI_4NL3/Project/project-4nl3/.venv/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Feature matrix shapes - train: (160000, 10000) val: (40000, 10000) test: (59999, 10000)


In [4]:
# Random Baseline Model
rng = np.random.default_rng(1)
classes = np.unique(y_train)
y_pred_random = rng.choice(classes, size=len(y_test))
acc_random = accuracy_score(y_test, y_pred_random)
print("Random baseline accuracy:", round(acc_random, 4))
print(classification_report(y_test, y_pred_random, zero_division=0))

Random baseline accuracy: 0.1017
              precision    recall  f1-score   support

           1       0.10      0.10      0.10      6000
           2       0.10      0.10      0.10      6000
           3       0.10      0.11      0.11      6000
           4       0.10      0.10      0.10      6000
           5       0.10      0.10      0.10      6000
           6       0.10      0.10      0.10      6000
           7       0.10      0.10      0.10      6000
           8       0.10      0.10      0.10      6000
           9       0.11      0.10      0.10      5999
          10       0.10      0.10      0.10      6000

    accuracy                           0.10     59999
   macro avg       0.10      0.10      0.10     59999
weighted avg       0.10      0.10      0.10     59999



In [5]:
# Majority Vote Baseline Model
from collections import Counter
majority_class = Counter(y_train).most_common(1)[0][0]
y_pred_majority = np.full(len(y_test), majority_class)
acc_majority = accuracy_score(y_test, y_pred_majority)
print("Majority baseline accuracy:", round(acc_majority, 4))
print("Predicted class for all:", majority_class)
print(classification_report(y_test, y_pred_majority, zero_division=0))

Majority baseline accuracy: 0.1
Predicted class for all: 8
              precision    recall  f1-score   support

           1       0.00      0.00      0.00      6000
           2       0.00      0.00      0.00      6000
           3       0.00      0.00      0.00      6000
           4       0.00      0.00      0.00      6000
           5       0.00      0.00      0.00      6000
           6       0.00      0.00      0.00      6000
           7       0.00      0.00      0.00      6000
           8       0.10      1.00      0.18      6000
           9       0.00      0.00      0.00      5999
          10       0.00      0.00      0.00      6000

    accuracy                           0.10     59999
   macro avg       0.01      0.10      0.02     59999
weighted avg       0.01      0.10      0.02     59999



In [6]:
# Logistical Regression Baseline Model 
lr = LogisticRegression(max_iter=500, C=1.0, solver="lbfgs", random_state=1)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print("Logistic Regression accuracy:", round(acc_lr, 4))
print(classification_report(y_test, y_pred_lr, zero_division=0))

Logistic Regression accuracy: 0.6461
              precision    recall  f1-score   support

           1       0.56      0.50      0.53      6000
           2       0.57      0.68      0.62      6000
           3       0.70      0.71      0.71      6000
           4       0.48      0.47      0.48      6000
           5       0.79      0.80      0.80      6000
           6       0.83      0.82      0.82      6000
           7       0.48      0.46      0.47      6000
           8       0.65      0.63      0.64      6000
           9       0.67      0.71      0.69      5999
          10       0.72      0.67      0.70      6000

    accuracy                           0.65     59999
   macro avg       0.65      0.65      0.65     59999
weighted avg       0.65      0.65      0.65     59999



In [7]:
# Random Forest Baseline Model
rf = RandomForestClassifier(n_estimators=100, max_depth=50, random_state=1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print("Random Forest accuracy:", round(acc_rf, 4))
print(classification_report(y_test, y_pred_rf, zero_division=0))

Random Forest accuracy: 0.5209
              precision    recall  f1-score   support

           1       0.62      0.36      0.46      6000
           2       0.25      0.76      0.37      6000
           3       0.60      0.53      0.56      6000
           4       0.59      0.28      0.38      6000
           5       0.66      0.74      0.70      6000
           6       0.78      0.57      0.66      6000
           7       0.61      0.30      0.40      6000
           8       0.69      0.41      0.51      6000
           9       0.56      0.71      0.62      5999
          10       0.69      0.56      0.62      6000

    accuracy                           0.52     59999
   macro avg       0.60      0.52      0.53     59999
weighted avg       0.60      0.52      0.53     59999



In [8]:
# Export testing label in model directory
def export_testing_label_in_model_dir(model_dir, y_pred):
    model_path = Path("results") / model_dir
    model_path.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"class_index": y_pred}).to_csv(
        model_path / "testing_label.csv",
        index=False
    )

export_testing_label_in_model_dir("random", y_pred_random)
export_testing_label_in_model_dir("majority", y_pred_majority)
export_testing_label_in_model_dir("logistic_regression", y_pred_lr)
export_testing_label_in_model_dir("random_forest", y_pred_rf)